In [1]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db_semantic")

collection = client.get_or_create_collection(
    name="website_embeddings_semantic2"
)
print("Stored embeddings in ChromaDB:", collection.count())

Stored embeddings in ChromaDB: 53


In [2]:
print(collection.metadata)

{'hnsw:space': 'cosine'}


In [3]:
import ollama
from sentence_transformers import CrossEncoder

reranker_model = CrossEncoder("BAAI/bge-reranker-base")

def get_chunks(text):
    response = ollama.embeddings(
    model="bge-m3:567m",
    prompt=text
    )

    query_embedding = response["embedding"]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=20
    )

    chunks = results["documents"][0]
    
    return chunks

def rerank(text, chunks, final_k=5):
    pairs = [
        [text, chunk]
        for chunk in chunks
        ]


    scores = reranker_model.predict(pairs)

    # Combine chunk + score
    ranked_results = list(zip(chunks, scores))

    # Sort by score descending
    ranked_results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    final_chunks = ranked_results[:5]

    context = "".join(
        f"\n--- Chunk {i} ---\n{chunk}\n"
        for i, (chunk, score) in enumerate(final_chunks, start=1)
    )
    
    return context


/home/puresitedesktop/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████| 201/201 [00:00<00:00, 7790.27it/s]


In [4]:
def generate_answer(query, context):
    prompt = f"""
    You are a company information assistant.
    use the following context to answer concisely

    Can the retrieved context answer this question?
    Return only YES or NO.
    if YES then answer the question.
    if NO then return "answer not found in the context"
    
    CONTEXT:
    {context}

    QUESTION:
    {query}

    FINAL ANSWER:
    """

    response = ollama.generate(
        model="qwen2.5:1.5b",
        prompt=prompt,
        stream=False,
        options={
            "temperature": 0,
            "top_p": 0.1,
            "repeat_penalty": 1.1
        }
    )

    answer = response["response"].strip()

    return answer

In [5]:
def self_rag(text):
    print("Query:", text)
    chunks = get_chunks(text)
    chunks = rerank(text, chunks, final_k=5)
    print("=====chunks=====",chunks)
    
    response = generate_answer(text, chunks)

    return response

In [6]:
text = "how to code in python?"
print(self_rag(text))

Query: how to code in python?
=====chunks===== 
--- Chunk 1 ---
## Creative Services
- Banners
- GIF
- Carousels
- QR codes interactive for CTV
- Wobble Creatives (HTML5)
- Interactive ad
- Video Ad Creation

## QA &amp; Troubleshooting
- Bug fixes and Bug resolution
- Performing root cause analysis
- Investigating and resolving code/tag issues
- DFP A-Z Troubleshooting
- Troubleshooting non-rendering ads
----

--- Chunk 2 ---
# Design Solutions
Transform your ideas into stunning designs with our expert design solutions

## UX/UI Solutions
Our team is dedicated to providing innovative and customized design solutions to make your product stand out. From wireframing and prototyping to usability testing and implementation, our team will guide you through every step of the design process. We work closely with our clients to understand their brand and target audience and create an intuitive and easy-to-use user interface. With our UI/UX design solutions, you can be confident that your produ

In [6]:
text = "What can this company help clients with?"

print(self_rag(text))

Query: What can this company help clients with?
=====chunks===== 
--- Chunk 1 ---
### Uptime
You let us know, we will be available. We work 24x7. We can even support on weekends too.

### Flexibility
Working on just the given task is not our motto, we prefer to help our client in all other possible tasks around the given tasks, because we understand your world.

### Client Services
Effective Client Services and well dedicated to reach you at all the times.
----

--- Chunk 2 ---
## Why the name Dollarbird?
How interesting is it to see the beautiful colors of the actual bird with the name Dollarbird? In the same way, we wanted to grow in such a manner that we also made our clients' lives colorful, we decided to fly to different levels along with our clients, hence the name DOLLARBIRD.

### How did it all start?
It all started with two employees, one client, and one service, always keeping the vision alive in the work we offered, and that has transformed Dollarbird into a company that has

In [10]:
text = "director of ads"

print(self_rag(text))

Query: director of ads
=====chunks===== 
--- Chunk 1 ---
### Jayanth N - Sr Director - Client Services

### Vikas M - Director - Ad Operations

### Amaresh Patil - Director of Technology

### Ravikiran S - Sr Manager of Technology
----

--- Chunk 2 ---
# Our Team  
### Chandrahasa S K - Founder &amp; CEO

### Sundar S - Partner &amp; COO

### Yogesh Jadhav - Chief Revenue Officer

### Shalom Charles - Director - Sales &amp; Marketing

### Robert Lutsky - Advisor of Sales, North America

--- Chunk 3 ---
# Our Offerings
We assist you in establishing brand connections with your target audience through digital and cost-effective marketing methods

## Social Media Management
- Strategy Building based on Trend
- Influencer Collaboration Strategy
- Brand awareness
- Engagement Growth
- Unique Content Poster Creation
- Analytics reporting
- Community Management
Regardless of the size of your company or the type of product or service you offer, we will provide you with a positive result and hel

In [8]:
text = "where this company is located"

print(self_rag(text))

Query: where this company is located
=====chunks===== 
--- Chunk 1 ---
# Locations
[India - 68K, Hootagalli Industrial Area, Mysore, Karnataka, India - 570018](https://goo.gl/maps/oQuyv7z3ZRWdDuBy5)
USA - 1201 Orange St #600 Wilmington, DE 19899
Israel - Tel Aviv, Israel

### Contact us
India
[+91 9611525444](tel:+91 9611525444)
[sales@dollarbirdinc.com](mailto:sales@dollarbirdinc.com)
USA
[+1 978-612-6344](tel:+1 978-612-6344)
[sales@dollarbirdinc.com](mailto:sales@dollarbirdinc.com)
Israel
[+972 54-674-4142](tel:+972 54-674-4142)
[sales@dollarbirdinc.com](mailto:sales@dollarbirdinc.com)
Copyright © 2026, Dollarbird | All Rights Reserved | [Privacy Policy](https://www.dollarbirdinc.com//privacy-policy/) | [Terms &amp; Conditions](https://www.dollarbirdinc.com/wp-content/uploads/2022/08/TERMS-OF-USE-1-1.pdf)
---------

--- Chunk 2 ---
# Our Team  
### Chandrahasa S K - Founder &amp; CEO

### Sundar S - Partner &amp; COO

### Yogesh Jadhav - Chief Revenue Officer

### Shalom Charles - D

In [9]:
text = "What services does this company provide?"

print(self_rag(text))

Query: What services does this company provide?
=====chunks===== 
--- Chunk 1 ---
# Services Overview
The company offers:
- Branding
- Design Solutions
- Digital Operations
- IT Services
- Sales Services
- Software Solutions
- Support Services
----

--- Chunk 2 ---
# Support Services
Maximize your revenue with our efficiency and timely execution of your core program or process

## Our Focus
We share a wide range of best quality customized services that help you to achieve your targeted custom goals in an efficient and accurate way. We are one who you look forward to grow your business in a customized way

## Our Offerings
We are here to take care of your needs, which is an integral part of any growing company

### Inbound and Outbound Support
- Highly Skilled Professionals
- Flexibility in Pricing
- Professional Agents
- Skill-base routing
- 24/7 support
- CRM integration
- Call and mail monitoring
- Reports
Looking for a service that treats its clients better?
It's right! We'll do it 

In [21]:
text = "how can you assist me?"

print(self_rag(text))

Query: how can you assist me?
=====chunks===== 
--- Chunk 1 ---
# Our Identity
You can count on us to help you achieve your goals and find solutions to your company's problems. With our team of specialists in Advertising, Software Development, Design, and Sales Support, we provide end-to-end solutions to all your needs. We can provide an ultimate solution by listening to your pain points.
----

--- Chunk 2 ---
# Our Focus
Whether you are an entrepreneur, a startup, or a small business owner, we provide a step-by-step branding process to ensure that each brand object aligns with the latest brand values and market trends while keeping the audience's needs and tastes in mind. To achieve greater levels of success, we assist you in establishing brand connections with your target audience through digital and cost-effective marketing methods
----

--- Chunk 3 ---
# Our Offerings
We assist you in establishing brand connections with your target audience through digital and cost-effective market